# Energy Output Prediction — AI Product Management Case Study
### Federico Silvera | Senior Product Manager | Duke University AI PM Specialization

---

> This notebook applies a Product Manager lens to a real ML regression problem: predicting the hourly power output of a Combined Cycle Power Plant (CCPP) based on environmental conditions.
>
> The goal is not to build the best model — it's to answer the question a PM actually faces: **which model should we ship, and why?**

## 1. Business Context & Objective

**Why does this problem matter?**

Combined Cycle Power Plants generate electricity by combining a gas turbine with a steam turbine. Their output is directly affected by ambient conditions — temperature, pressure, humidity, and exhaust vacuum — which fluctuate constantly.

- Overestimating output leads to supply shortages; underestimating leads to wasted capacity and cost.
- A reliable predictive model reduces operational uncertainty and enables better energy dispatch decisions.

**Who would use this model?**
- **Grid operators** — to anticipate available capacity and plan dispatch schedules.
- **Plant managers** — to benchmark actual vs. predicted output and flag anomalies.
- **Commercial teams** — to inform energy trading decisions based on forecasted output.

**Product Objective**

Reduce operational uncertainty in energy dispatch planning by providing a reliable hourly prediction of power plant output — accurate enough to inform real decisions, interpretable enough to build operator trust.

**Dataset:** 9,568 hourly records from a real CCPP over 6 years. Features: Ambient Temperature (AT), Exhaust Vacuum (V), Ambient Pressure (AP), Relative Humidity (RH). Target: Net hourly power output (PE) in MW.

## 2. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded successfully.')

## 3. Data Loading & Overview

In [ ]:
df = pd.read_csv('../data/ccpp_data.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nFeatures: {list(df.columns[:-1])}')
print(f'Target:   {df.columns[-1]}')
print('\n--- First 5 rows ---')
display(df.head())
print('\n--- Descriptive Statistics ---')
display(df.describe().round(2))
print('\n--- Missing Values ---')
print(df.isnull().sum())

## 4. Exploratory Data Analysis

Before training any model, we need to understand the data structure, variable distributions, and relationships between features and the target.

**Key question at this stage:** Which variables are most correlated with power output? Are there non-linear relationships that a simple linear model would miss?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation matrix
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Correlation Matrix', fontsize=13, fontweight='bold', pad=12)

# Distribution of target variable
axes[1].hist(df['PE'], bins=40, color='#1E3A5F', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of Power Output (PE)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Net Power Output (MW)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].axvline(df['PE'].mean(), color='#E74C3C', linestyle='--',
                label=f'Mean: {df["PE"].mean():.1f} MW')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Correlation with target (PE) ---')
print(df.corr()['PE'].sort_values(ascending=False).to_string())

## 5. Model Training

Two models are evaluated:
- **Linear Regression** — simple, interpretable baseline. Assumes linear relationships.
- **Random Forest Regressor** — captures non-linear relationships and feature interactions.

Both are evaluated using 5-fold cross-validation before final test set evaluation.

In [ ]:
# Define features and target
X = df[['AT', 'V', 'AP', 'RH']]
y = df['PE']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')

# Cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_cv(model, X_train, y_train, cv):
    mse_scores = -cross_val_score(model, X_train, y_train,
                                  cv=cv, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(mse_scores)
    r2_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
    return (mse_scores.mean(), mse_scores.std(),
            rmse_scores.mean(), rmse_scores.std(),
            r2_scores.mean(), r2_scores.std())

# Instantiate models
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation
print('\n--- Cross-Validation Results (5-fold) ---')
mse_lr, mse_lr_std, rmse_lr, rmse_lr_std, r2_lr, r2_lr_std = evaluate_cv(lr, X_train, y_train, kf)
mse_rf, mse_rf_std, rmse_rf, rmse_rf_std, r2_rf, r2_rf_std = evaluate_cv(rf, X_train, y_train, kf)

print(f'Linear Regression  — CV RMSE: {rmse_lr:.2f} ± {rmse_lr_std:.2f} | CV R²: {r2_lr:.3f} ± {r2_lr_std:.3f}')
print(f'Random Forest      — CV RMSE: {rmse_rf:.2f} ± {rmse_rf_std:.2f} | CV R²: {r2_rf:.3f} ± {r2_rf_std:.3f}')

# Train final models
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
print('\nModels trained on full training set.')

## 6. Model Evaluation

Final evaluation on the held-out test set. Cross-validation already confirmed generalization; test set gives the headline numbers.

In [ ]:
y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)

def evaluate_test(y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return mse, rmse, r2

mse_test_lr, rmse_test_lr, r2_test_lr = evaluate_test(y_test, y_pred_lr)
mse_test_rf, rmse_test_rf, r2_test_rf = evaluate_test(y_test, y_pred_rf)

print('--- Test Set Results ---')
print(f'\n{"Model":<22} {"MSE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 50)
print(f'{"Linear Regression":<22} {mse_test_lr:>8.2f} {rmse_test_lr:>8.2f} {r2_test_lr:>8.3f}')
print(f'{"Random Forest":<22} {mse_test_rf:>8.2f} {rmse_test_rf:>8.2f} {r2_test_rf:>8.3f}')
print(f'\nRMSE improvement (RF vs LR): {((rmse_test_lr - rmse_test_rf) / rmse_test_lr * 100):.1f}%')

# Actual vs Predicted plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest'],
    ['#2E86AB', '#1E3A5F']
):
    ax.scatter(y_test, y_pred, alpha=0.4, color=color, s=12)
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=1.5, label='Perfect prediction')
    ax.set_title(f'{title}: Actual vs Predicted', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Power Output (MW)', fontsize=10)
    ax.set_ylabel('Predicted Power Output (MW)', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Model Comparison: Predicted vs Actual Power Output',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance

**What drives power output?**

Feature importance answers a critical PM question: which variables does the model actually rely on? This matters for two reasons:
1. It makes the Random Forest less of a black box — we can explain *why* it makes a prediction.
2. It surfaces operational insights for the business: if one variable dominates, the forecasting workflow can be simplified around it.

In [ ]:
feature_names = ['Ambient Temp (AT)', 'Exhaust Vacuum (V)',
                 'Ambient Pressure (AP)', 'Relative Humidity (RH)']

importances = rf.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1E3A5F', '#2E86AB', '#A8D8EA', '#CAE4F1']
bars = ax.barh(
    [feature_names[i] for i in sorted_idx],
    importances[sorted_idx],
    color=colors
)

ax.set_xlabel('Feature Importance Score', fontsize=11)
ax.set_title('What drives power output? — Feature Importance (Random Forest)',
             fontsize=12, fontweight='bold', pad=15)

for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.1%}', va='center', fontsize=10)

ax.invert_yaxis()
ax.set_xlim(0, importances.max() * 1.15)
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('--- Business Interpretation ---')
for i in sorted_idx:
    print(f'  {feature_names[i]:<28} {importances[i]:.1%}')

**Business Interpretation:**

Ambient Temperature is the overwhelmingly dominant predictor (90.1% importance), dwarfing all other features combined. Exhaust Vacuum, Ambient Pressure, and Relative Humidity collectively account for less than 10% of the model's predictive power.

This is far stronger than typical thermodynamic intuition would suggest — it's not just that temperature matters, it's that it **nearly entirely drives output variability in this dataset**.

**For grid operators:** A reliable temperature forecast is almost sufficient on its own to anticipate generation capacity. This simplifies the forecasting workflow significantly.

**Modeling flag for production:** Such extreme dominance by a single feature warrants checking for **data leakage or collinearity** before deploying in production. It could reflect a genuine physical relationship — or a data artifact that would not hold on new data.

## 8. Model Trade-offs

This is the core PM decision: given two models with different accuracy/interpretability profiles, which one should we recommend — and why?

| | Linear Regression | Random Forest |
|---|---|---|
| **Test RMSE** | ~4.50 MW | ~3.24 MW |
| **Test R²** | ~0.930 | ~0.964 |
| **Interpretability** | High — coefficients are directly readable | Low by default — requires feature importance layer |
| **Non-linear relationships** | Cannot capture | Captures natively |
| **Production complexity** | Low | Medium (monitoring, versioning) |
| **Operator trust** | Easier to explain | Requires additional explainability tooling |

**Recommended decision:** Deploy Random Forest for operational predictions. Maintain Linear Regression as the interpretable audit layer.

This mirrors a common product pattern: use the performant model for automated decisions, keep the interpretable one for stakeholder explainability and trust-building. Random Forest *earns its complexity* — 30% lower RMSE over an already-strong baseline is meaningful in an operational context.

> As a PM who has worked at UTE (Uruguay's national electric utility), I recognize that operator trust is often the real adoption blocker — not model accuracy. A 30% RMSE improvement means nothing if dispatchers don't trust the output.

## 9. Product Takeaways

Working through this project as a PM — not as a data scientist — clarified a few things I now apply in my actual product work:

**1. The metric isn't the goal — the decision is.**
RMSE of 3.24 is meaningless without knowing the cost of a 3 MW prediction error. Before choosing a model, a PM needs to answer: *what happens if the model is wrong, and how wrong is too wrong?* That's a business question, not a statistical one.

**2. Interpretability is a product requirement, not a nice-to-have.**
At UTE, I saw firsthand that operators won't act on a system they don't understand. Random Forest outperforms Linear Regression on every metric — but without feature importance and confidence intervals, it may never get adopted.

**3. Extreme feature dominance is a signal, not just a result.**
A 90% importance score for a single feature (AT) simplifies the operational story — but it also raises a red flag. Before production, a PM should ask: is this a genuine physical relationship, or a data artifact? That question is as important as the accuracy numbers.

**4. Cross-validation matters more than test score for production conversations.**
A model that scores well on one test set might be overfit. CV RMSE of 3.46 ± 0.15 tells you the model generalizes — that's the number to bring to a production review.

**5. Questions a PM should ask before any model goes to production:**
- What's the cost of a false positive vs. false negative in this domain?
- Who owns model monitoring, and what triggers a retrain?
- How will users interact with predictions — dashboard, alert, API?
- Is there a regulatory requirement for explainability?
- What's the data refresh cadence, and does the model assume it's static?
- Does extreme feature dominance reflect the real world, or a data issue?

## 10. Risks & What's Missing for Production

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Data drift** | Plant conditions change over time; a model trained on historical data may degrade | Schedule periodic retraining; monitor RMSE in production |
| **Static dataset** | 9,568 records from a single plant — no temporal split was applied | Treat results as directional, not as production-ready validation |
| **Extreme feature dominance** | AT at 90.1% importance warrants checking for data leakage or collinearity | Validate with domain experts and test on holdout time periods |
| **No real error cost analysis** | RMSE treats all errors equally; a 10 MW error at peak demand may cost 10× a 10 MW error at night | Define a cost-weighted error function before production |
| **Single plant generalization** | Model was trained on one CCPP; performance on different plants is unknown | Retrain per plant or validate on holdout plant data |
| **No production validation** | Model has not been tested on live data or in an operational environment | A pilot with shadow deployment is required before go-live |

---

**About this case study:**
Built as part of Duke University's AI Product Management Specialization.
Federico Silvera — Senior Product Manager | [GitHub](https://github.com/fedesilverauy)